In [9]:
!python -m pip install langchain-google-genai python-dotenv


[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()


True

In [20]:
llm=ChatGoogleGenerativeAI(model="gemma-4-31b-it",temperature=0)

with open("bloodwork.txt","r") as f:
    blood_report=f.read()
    
print(blood_report[:200])

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5â€“17.5)
Hematocrit:        44%          


In [21]:

#first i have to extract prompt fromt the blood report and dive some instructions to llm to print like this
extraction_prompt = f"""
You are a medical data extraction assistant.

From the blood report below, extract ALL test values and classify each one as HIGH, LOW, or NORMAL 
based on the reference ranges provided in the report.

Format your response as:
- Test Name: value | Status: HIGH/LOW/NORMAL | Reference: range

Blood Report:
{blood_report}
"""

response=llm.invoke(extraction_prompt)

print("=== STAGE 1: EXTRACTED VALUES ===")
print(response.text)

- Hemoglobin: 15.1 g/dL | Status: NORMAL | Reference: 13.5–17.5
- Hematocrit: 44% | Status: NORMAL | Reference: 41–53%
- WBC: 6.8 x10^3/uL | Status: NORMAL | Reference: 4.5–11.0
- Platelets: 220 x10^3/uL | Status: NORMAL | Reference: 150–400
- Total Cholesterol: 238 mg/dL | Status: HIGH | Reference: <200
- LDL Cholesterol: 162 mg/dL | Status: HIGH | Reference: <100
- HDL Cholesterol: 36 mg/dL | Status: LOW | Reference: >40
- Triglycerides: 188 mg/dL | Status: HIGH | Reference: <150
- Glucose (Fasting): 92 mg/dL | Status: NORMAL | Reference: 70–99
- HbA1c: 5.3% | Status: NORMAL | Reference: <5.7%
- Creatinine: 1.0 mg/dL | Status: NORMAL | Reference: 0.7–1.3
- eGFR: 82 mL/min | Status: NORMAL | Reference: >60
- ALT: 28 U/L | Status: NORMAL | Reference: 7–40
- AST: 25 U/L | Status: NORMAL | Reference: 10–40
- Bilirubin Total: 0.8 mg/dL | Status: NORMAL | Reference: 0.2–1.2


In [22]:
diet_prompt = f"""
You are a clinical nutritionist specializing in Indian dietary habits.

Based on the blood work analysis below, write:
1. A short health summary in 4-5 lines explaining the patient's condition in simple language
2. A short, practical Indian diet plan having only two sections (1) Foods to avoid (2) Foods to eat more of. 
   Do not include any other sections in diet plan.

Blood Work Analysis:
{extraction_prompt}
"""

response=llm.invoke(diet_prompt)

print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(response.text)

=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===
**Health Summary**
Your blood reports indicate that your blood sugar, kidney, and liver functions are all within the normal range. However, you have high levels of total cholesterol, LDL (bad cholesterol), and triglycerides, along with low HDL (good cholesterol). This condition, known as dyslipidemia, increases the risk of heart-related issues. Improving your diet and increasing physical activity will help balance these lipid levels.

**Indian Diet Plan**

**1. Foods to Avoid**
*   **Deep-fried snacks:** Samosas, pakoras, puri, and namkeens.
*   **Saturated & Trans Fats:** Excessive ghee, butter, vanaspati/dalda, and palm oil.
*   **Refined Carbohydrates:** Maida (white flour), white bread, biscuits, and sugary sweets/mithai.
*   **Full-Fat Dairy:** Full-cream milk, malai, and processed cheese.
*   **Processed Meats:** Red meat and processed sausages or deli meats.

**2. Foods to Eat More Of**
*   **Whole Grains:** Oats, dalia, brown rice, an

In [23]:
 # WORKFLOW is the first category to call ai
# prompt -> LLM (Extracted text) -> extracted values i get -> llm(generate diest plan) -> diet plan
# here i call llm for 2 times ,nothing is autonoomus
#this is not an agentic ai...its just a gen ai or an workflow

#streamlit is a way to create a quick ui